# Flickr8k Baseline Runner (Google Colab + Google Drive + GitHub)
**Project:** *A Controlled Evaluation of Patch-Preserving and Query-Resampling Connectors for Resource-Constrained Image Captioning*  
**Group 13:** Huỳnh Trần Uy (23125047), Trần Mỹ An (23125049), Nguyễn Hoàng Gia Hân (23125055)

### Workflow
1. **Code** lives on GitHub (`https://github.com/Uyhuynh23/Group13_CV.git`). Whenever you edit code locally and `git push`, just re-run **Step 2** (`!git pull`) in this notebook.
2. **Data, Cached CLIP Tensors, Checkpoints, and Evaluation Logs** live on your **Google Drive** (`/content/drive/MyDrive/Group13_CV_Data`) so you never lose progress when Colab disconnects.

## Step 1: Mount Google Drive & Verify Tesla T4 GPU

In [ ]:
import os
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Create persistent directories on Google Drive
DRIVE_ROOT = '/content/drive/MyDrive/Group13_CV_Data'
ZIP_CACHE_DIR = f'{DRIVE_ROOT}/flickr8k_zips'
CLIP_CACHE_DIR = f'{DRIVE_ROOT}/cache'
CKPT_DIR = f'{DRIVE_ROOT}/checkpoints'
RESULTS_DIR = f'{DRIVE_ROOT}/results'
LOCAL_DATA_DIR = '/content/flickr8k_local'

for d in [DRIVE_ROOT, ZIP_CACHE_DIR, CLIP_CACHE_DIR, CKPT_DIR, RESULTS_DIR, LOCAL_DATA_DIR]:
    os.makedirs(d, exist_ok=True)

print('Persistent Google Drive root:', DRIVE_ROOT)
!nvidia-smi

## Step 2: Clone / Pull GitHub Repository & Install Dependencies
Re-run the cell below anytime you push code updates to GitHub.

In [ ]:
REPO_URL = 'https://github.com/Uyhuynh23/Group13_CV.git'
REPO_DIR = '/content/Group13_CV'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin main

%cd {REPO_DIR}
!pip install -q -r requirements.txt

## Step 3: Auto-Download Flickr8k to Google Drive & Cache CLIP ViT-B/32 Features
- Automatically downloads `Flickr8k_Dataset.zip` (1.1 GB) and `Flickr8k_text.zip` directly into your Google Drive (`flickr8k_zips/`) so you never have to upload 1 GB manually.
- Extracts both **1×512 global CLIP features** (for the sanity baseline) and **49×768 patch tokens in FP16** (~0.60 GB total, for the upcoming 2×2 study) into `train_cache.pt`, `val_cache.pt`, and `test_cache.pt` on Google Drive.
- **Note:** Once `train_cache.pt`, `val_cache.pt`, and `test_cache.pt` exist on your Google Drive, future sessions skip downloading and feature extraction automatically!

In [ ]:
%cd /content/Group13_CV

expected_caches = [
    f'{CLIP_CACHE_DIR}/train_cache.pt',
    f'{CLIP_CACHE_DIR}/val_cache.pt',
    f'{CLIP_CACHE_DIR}/test_cache.pt',
]

if all(os.path.isfile(p) for p in expected_caches):
    print('[SKIP] All CLIP feature caches already exist on Google Drive!')
    for p in expected_caches:
        print(f'  - {p} ({os.path.getsize(p)/(1024**2):.1f} MB)')
else:
    # 1. Download Flickr8k zips to Google Drive and extract locally for fast I/O
    !python download_flickr8k.py --data_dir {LOCAL_DATA_DIR} --zip_cache_dir {ZIP_CACHE_DIR}
    
    # 2. Extract and cache frozen CLIP ViT-B/32 features (FP16) onto Google Drive
    !python parse_flickr8k.py \
        --data_dir {LOCAL_DATA_DIR} \
        --splits_dir ./data/flickr8k/splits \
        --cache_dir {CLIP_CACHE_DIR} \
        --batch_size 64

## Step 4: Train the Pipeline Sanity Baseline (`global_linear`)
Implements Proposal Section IV.D & IV.E:
- **Architecture:** Global CLIP ViT-B/32 feature (`512`-d) + Linear projection to GPT-2 prefix (`1 × 768`), with frozen GPT-2 Small decoder.
- **Checkpoint selection:** Saved to Google Drive (`checkpoints/baseline_global_linear_seed42_best.pt`) based on validation CIDEr.

In [ ]:
%cd /content/Group13_CV

!python train_flickr8k.py \
    --cache_dir {CLIP_CACHE_DIR} \
    --out_dir {CKPT_DIR} \
    --run_name baseline_global_linear_seed42 \
    --connector_type global_linear \
    --prefix_length 1 \
    --epochs 10 \
    --bs 64 \
    --lr 2e-4 \
    --patience 4 \
    --seed 42

### (Optional) Train Original ClipCap MLP Baseline (`clipcap_mlp`, `prefix_length=10`)

In [ ]:
# Uncomment to also run the 10-token ClipCap MLP baseline:
# !python train_flickr8k.py \
#     --cache_dir {CLIP_CACHE_DIR} \
#     --out_dir {CKPT_DIR} \
#     --run_name baseline_clipcap_mlp_seed42 \
#     --connector_type clipcap_mlp \
#     --prefix_length 10 \
#     --epochs 10 \
#     --bs 64 \
#     --lr 2e-4 \
#     --patience 4 \
#     --seed 42

## Step 5: Evaluate Baseline on Flickr8k Test Split (1,000 Images)
Computes **CIDEr, BLEU-1, BLEU-4, METEOR, ROUGE-L**, plus decoding latency (`ms/image`) and `tokens/sec` using deterministic Beam Search (`beam_size=5`) and Greedy Decoding (`beam_size=1`).

In [ ]:
%cd /content/Group13_CV

BEST_CKPT = f'{CKPT_DIR}/baseline_global_linear_seed42_best.pt'
TEST_CACHE = f'{CLIP_CACHE_DIR}/test_cache.pt'

# 1. Deterministic Beam Search (beam_size = 5)
!python evaluate.py \
    --checkpoint {BEST_CKPT} \
    --cache_path {TEST_CACHE} \
    --out_dir {RESULTS_DIR} \
    --beam_size 5 \
    --max_new_tokens 35

# 2. Greedy Decoding Robustness Check (beam_size = 1)
!python evaluate.py \
    --checkpoint {BEST_CKPT} \
    --cache_path {TEST_CACHE} \
    --out_dir {RESULTS_DIR} \
    --beam_size 1 \
    --max_new_tokens 35

## Step 6: Inspect Qualitative Predictions & Training Summary

In [ ]:
import json

eval_file = f'{RESULTS_DIR}/eval_test_baseline_global_linear_seed42_best_beam5.json'
with open(eval_file, 'r') as f:
    report = json.load(f)

print('=== SAMPLE GENERATED CAPTIONS (TEST SET) ===\n')
for item in report['predictions'][:5]:
    print(f"Image ID   : {item['image_id']}")
    print(f"Prediction : {item['prediction']}")
    print(f"Reference 1: {item['references'][0]}")
    print(f"Reference 2: {item['references'][1]}")
    print('-' * 60)